# 🎬 Faceless Review Video Generator — 1-Click Colab

**Chỉ cần chạy cell duy nhất bên dưới.**

---
⚠️ **Runtime**: `Runtime` → `Change runtime type` → Chọn **T4 GPU**
---

In [ ]:
# ============================================================
# 🚀 1-CLICK FACELESS REVIEW VIDEO GENERATOR
# ============================================================
# Chỉ cần ấn PLAY — mọi thứ tự động chạy!
# ============================================================

import os, sys, subprocess, json, time, ipywidgets
from IPython.display import display, HTML, clear_output
from google.colab import output

# ============================================================
# BƯỚC 0: Cấu hình — Edit các dòng này nếu cần
# ============================================================

TOPIC             = "Đánh giá iPhone 15 Pro Max"   # Chủ đề review
VIDEO_MODEL       = "wan_2_1"                      # wan_2_1 hoặc ltx_video
OPENROUTER_MODEL  = "deepseek/deepseek-r1:free"   # Model OpenRouter
TTS_VOICE         = "vi-VN-HoaiMyNeural"          # Giọng đọc
LAUNCH_GRADIO     = False                          # True = mở Web UI thay vì CLI

# ============================================================
# BẮT ĐẦU
# ============================================================

print("=" * 60)
print("🎬 FACELESS REVIEW VIDEO GENERATOR — 1-CLICK")
print("=" * 60)

# --- 1. Cài đặt dependencies ---
print("\n[1/5] 📦 Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "gradio>=4.0.0", "requests", "diffusers", "transformers",
    "accelerate", "torch", "torchvision", "edge-tts",
    "moviepy", "ffmpeg-python", "imageio-ffmpeg", "pillow", "numpy",
], capture_output=True)
print("   ✅ Dependencies installed")

# --- 2. Clone repo (nếu chưa có) ---
print("\n[2/5] 📂 Cloning repository...")
REPO_DIR = "/content/AllInOne"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/Motchucuncon/AllInOne.git", REPO_DIR], check=True)
    print("   ✅ Repository cloned")
else:
    print("   ✅ Repository already exists")

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"   📁 Working dir: {os.getcwd()}")

# --- 3. Nhập API key ---
print("\n[3/5] 🔑 Configuring OpenRouter API key...")
from getpass import getpass
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    API_KEY = getpass("   Nhập OpenRouter API key (https://openrouter.ai/keys): ")
    os.environ["OPENROUTER_API_KEY"] = API_KEY
print("   ✅ API key set")

# --- 4. Chạy pipeline hoặc UI ---
if LAUNCH_GRADIO:
    print("\n[4/5] 🚀 Launching Gradio Web UI...")
    print("   ⏳ Đợi 10 giây để Gradio khởi động...")
    import logging
    logging.getLogger('gradio').setLevel(logging.ERROR)
    from app import build_ui
    demo = build_ui()
    # Chạy trong thread riêng
    import threading
    def run_ui():
        demo.launch(share=True, debug=False, server_name="0.0.0.0", server_port=7860, prevent_thread_lock=True)
    t = threading.Thread(target=run_ui, daemon=True)
    t.start()
    time.sleep(8)
    display(HTML("<p>✅ Gradio UI đang chạy! Click link bên dưới:</p>"))
    print(f"   📎 Gradio Share URL sẽ hiện ở cell output phía trên")
else:
    print("\n[4/5] 🎬 Running pipeline...")
    from core.script_gen import generate_storyboard
    from core.audio_gen import generate_audio_and_subtitles
    from core.image_gen import generate_broll_images
    from core.video_gen import render_video_clips
    from core.composer import compose_final_video

    # --- Storyboard ---
    print("\n   🤖 STEP 1/5: Storyboard generation...")
    storyboard = generate_storyboard(topic=TOPIC, model=OPENROUTER_MODEL, api_key=API_KEY)
    with open("output/storyboard.json", "w") as f:
        json.dump(storyboard, f, ensure_ascii=False, indent=2)
    scenes = storyboard.get("storyboard_scenes", [])
    print(f"      ✅ {len(scenes)} scenes generated")

    # --- Audio ---
    print("   🔊 STEP 2/5: Audio generation...")
    audio = generate_audio_and_subtitles(storyboard=storyboard, voice=TTS_VOICE, output_dir="output")
    print(f"      ✅ Audio: {audio['duration_seconds']:.1f}s")

    # --- Images ---
    print("   🖼️  STEP 3/5: B-roll image generation...")
    prompts = [s["broll_prompt"] for s in scenes]
    ids = [s["scene_id"] for s in scenes]
    images = generate_broll_images(prompts=prompts, scene_ids=ids, output_dir="output/images", unload_after=True)
    print(f"      ✅ {len(images)} images generated")

    # --- Video ---
    print(f"   🎬 STEP 4/5: Video rendering ({VIDEO_MODEL})...")
    clips = render_video_clips(image_paths=images, prompts=prompts, scene_ids=ids, model=VIDEO_MODEL, output_dir="output/clips")
    print(f"      ✅ {len(clips)} clips rendered")

    # --- Compose ---
    print("   🎞️  STEP 5/5: Composing final video...")
    final = compose_final_video(clip_paths=clips, audio_path=audio["audio_path"], subtitles_path=audio["subtitles_path"], output_dir="output")
    print(f"      ✅ Final video: {final}")

    # --- Hiển thị kết quả ---
    clear_output(wait=True)
    display(HTML(f"""
    <div style="background:#1a1a2e;padding:20px;border-radius:10px;color:white;font-family:sans-serif;">
        <h2>🎉 PIPELINE COMPLETE!</h2>
        <hr style="border-color:#333">
        <p>📝 <b>Topic:</b> {TOPIC}</p>
        <p>🎬 <b>Video model:</b> {VIDEO_MODEL}</p>
        <p>📋 <b>Scenes:</b> {len(scenes)}</p>
        <p>⏱️ <b>Duration:</b> {audio['duration_seconds']:.1f}s</p>
        <p>📁 <b>Output:</b> {final}</p>
        <hr style="border-color:#333">
        <p>📥 <b>Download:</b> Chạy cell bên dưới để tải video về</p>
    </div>
    """))

print("\n" + "=" * 60)
print("🎉 HOÀN THÀNH!")
print("=" * 60)

## 📥 Tải kết quả về máy (tuỳ chọn)

In [ ]:
from google.colab import files
from IPython.display import Video

final_path = "/content/AllInOne/output/final_review_video.mp4"

if os.path.exists(final_path):
    display(Video(final_path, width=640))
    files.download(final_path)
else:
    print("❌ Chưa có video. Chạy cell trên trước.")

# Hoặc download toàn bộ output/
!zip -j /content/output_files.zip /content/AllInOne/output/*
files.download("/content/output_files.zip")